# Face 04 · Distil into MobileFaceNet (on device)

**Student:** MobileFaceNet (≈1 M params, ≈4 MB fp16 / 1 MB int8, 112×112 → 512-d).
**Loss:** AdaFace (same labels) + **cosine embedding distillation** to the IResNet-50 teacher, so phone and server
embeddings live in the same space (the server repeats the 1:1 match – spec section 11.2).
Target: on-device embedding < 10 MB, < 50 ms on mid-range phones.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Face_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"

def conv_bn(i, o, k=3, s=1, p=1, g=1, act=True):
    layers = [nn.Conv2d(i, o, k, s, p, groups=g, bias=False), nn.BatchNorm2d(o)]
    return nn.Sequential(*layers, nn.PReLU(o)) if act else nn.Sequential(*layers)

class Bottleneck(nn.Module):
    def __init__(self, i, o, s, t=2):
        super().__init__()
        self.res = s == 1 and i == o
        self.conv = nn.Sequential(conv_bn(i, i * t, 1, 1, 0), conv_bn(i * t, i * t, 3, s, 1, g=i * t), conv_bn(i * t, o, 1, 1, 0, act=False))
    def forward(self, x):
        return x + self.conv(x) if self.res else self.conv(x)

class MobileFaceNet(nn.Module):
    def __init__(self, dim=512):
        super().__init__()
        cfg = [(64, 5, 2), (128, 1, 2), (128, 6, 1), (128, 1, 2), (128, 2, 1)]
        layers, inp = [conv_bn(3, 64, 3, 2, 1), conv_bn(64, 64, 3, 1, 1, g=64)], 64
        for out, n, s in cfg:
            for j in range(n):
                layers.append(Bottleneck(inp, out, s if j == 0 else 1, t=4 if out == 64 else 2)); inp = out
        self.features = nn.Sequential(*layers, conv_bn(128, 512, 1, 1, 0))
        self.gdc = conv_bn(512, 512, 7, 1, 0, g=512, act=False)       # global depthwise conv
        self.out = nn.Sequential(nn.Flatten(), nn.Linear(512, dim, bias=False), nn.BatchNorm1d(dim))
    def forward(self, x):
        return self.out(self.gdc(self.features(x)))

student = MobileFaceNet().to(device)
print(sum(p.numel() for p in student.parameters()) / 1e6, "M params")

In [ ]:
# teacher = IResNet(); teacher.load_state_dict(torch.load(RUNS / "iresnet50_adaface.pt")); teacher.eval().to(device)
# head = AdaFace(512, num_classes).to(device)
LAMBDA_KD = 4.0
def distill_loss(student_emb, teacher_emb, logits, labels):
    kd = 1 - F.cosine_similarity(F.normalize(student_emb), F.normalize(teacher_emb)).mean()
    return F.cross_entropy(logits, labels) + LAMBDA_KD * kd
# Training loop = notebook 03 loop, with: with torch.no_grad(): t = teacher(images)